# Recommender Creatin

In [ ]:
import pandas as pd
import numpy as np
import gc
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the datasets
interactions = pd.read_csv('https://raw.githubusercontent.com/ElPatron100/Library-Recommender-Team-Migros/main/interactions_train.csv')
interactions = interactions.rename(columns={'u': 'UserID', 'i': 'ItemID', 't':'timestamp'})
items = pd.read_csv("https://raw.githubusercontent.com/ElPatron100/Library-Recommender-Team-Migros/main/items.csv")
items = items.rename(columns={'i': 'ItemID'})
# Display the first rows of each dataset
display(interactions.head())
display(items.head())

,UserID,ItemID,timestamp
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09


,Title,Author,ISBN Valid,Publisher,Subjects,ItemID
0,Classification décimale universelle : édition ...,NaN,9782871303336; 2871303339,Ed du CEFAL,Classification décimale universelle; Indexatio...,0
1,Les interactions dans l'enseignement des langu...,"Cicurel, Francine, 1947-",9782278058327; 2278058320,Didier,didactique--langue étrangère - enseignement; d...,1
2,Histoire de vie et recherche biographique : pe...,NaN,2343190194; 9782343190198,L'Harmattan,Histoires de vie en sociologie; Sciences socia...,2
3,Ce livre devrait me permettre de résoudre le c...,"Mazas, Sylvain, 1980-",9782365350020; 236535002X; 9782365350488; 2365...,Vraoum!,Moyen-Orient; Bandes dessinées autobiographiqu...,3
4,Les années glorieuses : roman /,"Lemaitre, Pierre, 1951-",9782702180815; 2702180817; 9782702183618; 2702...,Calmann-Lévy,France--1945-1975; Roman historique; Roman fra...,4


In [ ]:
#Let’s check how many unique users and items we have in this dataset.
n_users = interactions.UserID.nunique()
n_items = interactions.ItemID.nunique()
print(f'Number of users = {n_users}, \n Number of books = {n_items} \n Number of interactions = {len(interactions)}')


Number of users = 7838, 
 Number of books = 15109 
 Number of interactions = 87047


## Split the Data into Training and Test Sets

In [ ]:
# let's first sort the interactions by user and time stamp
interactions = interactions.sort_values(["UserID", "timestamp"])
interactions.head(10)

,UserID,ItemID,timestamp
21035,0,0,1.680191e+09
28842,0,1,1.680783e+09
3958,0,2,1.680801e+09
29592,0,3,1.683715e+09
6371,0,3,1.683715e+09
41220,0,4,1.686569e+09
12217,0,5,1.687014e+09
19703,0,6,1.687014e+09
64522,0,7,1.687014e+09
29380,0,8,1.687260e+09


In [ ]:
#Next we can use the percentage rank from pandas to get a proportional ranking of the timestamps for each user.
interactions["pct_rank"] = interactions.groupby("UserID")["timestamp"].rank(pct=True, method='dense')
interactions.reset_index(inplace=True, drop=True)
interactions.head(10)

,UserID,ItemID,timestamp,pct_rank
0,0,0,1.680191e+09,0.04
1,0,1,1.680783e+09,0.08
2,0,2,1.680801e+09,0.12
3,0,3,1.683715e+09,0.16
4,0,3,1.683715e+09,0.20
5,0,4,1.686569e+09,0.24
6,0,5,1.687014e+09,0.28
7,0,6,1.687014e+09,0.32
8,0,7,1.687014e+09,0.36
9,0,8,1.687260e+09,0.40


In [ ]:
#Now all that remains is to pick the first 80% of each user's interactions for the training set and
# the rest for the test set. We can do so using the pct_rank column.
train_data = interactions[interactions["pct_rank"] <= 1]
test_data = interactions[interactions["pct_rank"] >= 0.8]
print("Training set size:", train_data.shape[0])
print("Testing set size:", test_data.shape[0])

Training set size: 87047
Testing set size: 21628


## Creating User-Item Matrices for Implicit Feedback

In [ ]:
print('number of users =', n_users, '| number of books =', n_items)

number of users = 7838 | number of books = 15109


In [ ]:
#define a function that creates the user-item data matrix. Each matrix cell will contain a 1 if there was an interaction and a 0 otherwise.
import numpy as np

# Define a function to create the data matrix
def create_data_matrix(data, n_users, n_items):
    """
    This function returns a numpy matrix with shape (n_users, n_items).
    Each entry is a binary value indicating positive interaction.
    """
    data_matrix = np.zeros((n_users, n_items))
    data_matrix[data["UserID"].values, data["ItemID"].values] = 1
    return data_matrix


In [ ]:
#use the function to create matrices for both the training and testing data

# Recalculate n_items to ensure it accommodates the maximum ItemID as an index
n_items = interactions.ItemID.max() + 1

# Create the training and testing matrices
train_data_matrix = create_data_matrix(train_data, n_users, n_items)
test_data_matrix = create_data_matrix(test_data, n_users, n_items)

# Display the matrices to understand their structure
print('train_data_matrix')
print(train_data_matrix)
print("number of non-zero values: ", np.sum(train_data_matrix))
print('test_data_matrix')
print(test_data_matrix)
print("number of non-zero values: ", np.sum(test_data_matrix))

train_data_matrix
[[1. 1. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
number of non-zero values:  64003.0
test_data_matrix
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
number of non-zero values:  19409.0


# Developping Function Useful Later

## Precision evaluation

In [ ]:
import numpy as np

# TODO: Implement the precision_recall_at_k function
def precision_recall_at_k(prediction, ground_truth, k=10):
    """
    Calculates Precision@K and Recall@K for top-K recommendations.
    Parameters:
        prediction (numpy array): The predicted interaction matrix with scores.
        ground_truth (numpy array): The ground truth interaction matrix (binary).
        k (int): Number of top recommendations to consider.
    Returns:
        precision_at_k (float): The average precision@K over all users.
        recall_at_k (float): The average recall@K over all users.
    """
    num_users = prediction.shape[0]
    precision_at_k, recall_at_k = 0, 0

    for user in range(num_users):
        # TODO: Get the indices of the top-K items for the user based on predicted scores
        top_k_items = np.argsort(prediction[user, :])[-k:]

        # TODO: Calculate the number of relevant items in the top-K items for the user
        relevant_items_in_top_k = np.isin(top_k_items, np.where(ground_truth[user, :] == 1)[0]).sum()

        # TODO: Calculate the total number of relevant items for the user
        total_relevant_items = ground_truth[user, :].sum()

        # Precision@K and Recall@K for this user
        precision_at_k += relevant_items_in_top_k / k
        recall_at_k += relevant_items_in_top_k / total_relevant_items if total_relevant_items > 0 else 0

    # Average Precision@K and Recall@K over all users
    precision_at_k /= num_users
    recall_at_k /= num_users

    return precision_at_k, recall_at_k


## Get the top 10 recommendations

In [ ]:
# Function to get top N recommendations for each user
def get_top_n_recommendations(prediction_matrix, n=10):
    top_n_recs = []
    for user_id in range(prediction_matrix.shape[0]):
        # Get the indices of items with the highest prediction scores for the current user
        # Use argsort to get indices that would sort the array, then take the last 'n' for top scores
        user_item_scores = prediction_matrix[user_id, :]
        top_item_indices = np.argsort(user_item_scores)[::-1][:n]

        for rank, item_id in enumerate(top_item_indices):
            top_n_recs.append({
                'UserID': user_id,
                'ItemID': item_id,
                'Predicted_Score': user_item_scores[item_id],
                'Rank': rank + 1
            })
    return pd.DataFrame(top_n_recs)

## Submission Format

In [ ]:
import pandas as pd

def create_submission_format(df, filename):
    # 1. Group by UserID and join ItemIDs with a space
    # We sort by Rank first to ensure the top-10 order is preserved
    submission_df = (
        df.sort_values(['UserID', 'Rank'])
        .groupby('UserID')['ItemID']
        .apply(lambda x: ' '.join(map(str, x)))
        .reset_index()
    )

    # 2. Rename columns to match sample_submission.csv
    submission_df.columns = ['user_id', 'recommendation']

    # 3. Save to CSV
    submission_df.to_csv(filename, index=False)
    print(f"Successfully created: {filename}")
    return submission_df

# Generate the two submission files



# Item-to-Item Recommender

In [ ]:
import numpy as np
import gc

def compute_jaccard_similarity(train_matrix):
    """
    Computes the Jaccard Similarity matrix for items.
    J(A, B) = |A ∩ B| / |A ∪ B|

    Parameters:
        train_matrix (numpy array): Binary user-item matrix (n_users x n_items).
    Returns:
        numpy array: Item-item similarity matrix (float32).
    """
    # 1. Ensure matrix is float32 for RAM efficiency [cite: 361]
    matrix = train_matrix.astype(np.float32)

    # 2. Intersection: Number of users who rented both Item A and Item B
    # This is equivalent to the dot product of the item columns
    print("Calculating intersection matrix...")
    intersection = matrix.T.dot(matrix)

    # 3. Get the sum of rentals for each item (the "size" of each set)
    item_sums = matrix.sum(axis=0)

    # 4. Union: Size(A) + Size(B) - Intersection(A, B)
    # Using broadcasting to create a matrix of (sums_i + sums_j)
    print("Calculating union matrix...")
    union = item_sums[:, np.newaxis] + item_sums[np.newaxis, :] - intersection

    # 5. Jaccard = Intersection / Union
    # We add a tiny epsilon to the denominator to avoid division by zero
    print("Computing final Jaccard scores...")
    jaccard_matrix = intersection / (union + 1e-9)

    # Cleanup intermediate large matrices to free RAM [cite: 440]
    del intersection, union, item_sums
    gc.collect()

    return jaccard_matrix


In [ ]:

# --- Execution ---
# Replace your previous item_similarity line with this:
item_similarity = compute_jaccard_similarity(train_data_matrix)

print(f"Jaccard Similarity Matrix Shape: {item_similarity.shape}")

Calculating intersection matrix...
Calculating union matrix...
Computing final Jaccard scores...
Jaccard Similarity Matrix Shape: (15291, 15291)


In [ ]:
#Predict Positive Interactions Using Item Similarity
import numpy as np

# Define the function to predict interactions based on item similarity
def item_based_predict(interactions, similarity, epsilon=1e-9):

  '''
  Predicts user-item interactions based on item-item similarity.
  Parameters:
  interactions (numpy array): The user-item interaction matrix.
  similarity (numpy array): The item-item similarity matrix.
  epsilon (float): Small constant added to the denominator to avoid division by zero.
  Returns:
  numpy array: The predicted interaction scores for each user-item pair.
  '''
  # np.dot does the matrix multiplication. Here we are calculating the
  # weighted sum of interactions based on item similarity
  pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon)
  return pred.T # Transpose to get users as rows and items as columns

# Calculate the item-based predictions for positive interactions
item_prediction = item_based_predict(train_data_matrix, item_similarity)

del item_similarity
gc.collect()


print("Predicted Interaction Matrix:")
print(item_prediction)
print(item_prediction.shape)

Predicted Interaction Matrix:
[[0.53777582 0.875562   0.73510189 ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.02402074 0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]
(7838, 15291)


In [ ]:
# Precision
precision_item_k, recall_item_k = precision_recall_at_k(item_prediction, test_data_matrix, k=10)

# Get top 10 recommendations for item-based CF
item_based_recs_df = get_top_n_recommendations(item_prediction, n=10)
item_based_recs_df['Recommender_Type'] = 'Item-Based'


#Submission Format
item_sub = create_submission_format(item_based_recs_df, 'item_cf_submission.csv')

precision_item_k, recall_item_k


Successfully created: item_cf_submission.csv


(np.float64(0.17825976014288777), np.float64(0.9002259957792488))

# User-to-User Recommender

In [ ]:
import numpy as np
import gc

def compute_user_jaccard_similarity(train_matrix):
    """
    Computes the Jaccard Similarity matrix for users.
    J(U1, U2) = |Items(U1) ∩ Items(U2)| / |Items(U1) ∪ Items(U2)|

    Parameters:
        train_matrix (numpy array): Binary user-item matrix (n_users x n_items).
    Returns:
        numpy array: User-user similarity matrix (float32).
    """
    # 1. Ensure matrix is float32 for RAM efficiency [cite: 361]
    matrix = train_matrix.astype(np.float32)

    # 2. Intersection: Number of shared books between User A and User B [cite: 539]
    # For user-user, we take the dot product of the matrix with its own transpose
    print("Calculating user intersection matrix...")
    intersection = matrix.dot(matrix.T)

    # 3. Get the total number of books rented by each user [cite: 540]
    user_sums = matrix.sum(axis=1)

    # 4. Union: Books(A) + Books(B) - Intersection(A, B) [cite: 546]
    print("Calculating user union matrix...")
    union = user_sums[:, np.newaxis] + user_sums[np.newaxis, :] - intersection

    # 5. Jaccard = Intersection / Union
    print("Computing final User Jaccard scores...")
    user_jaccard_matrix = intersection / (union + 1e-9)

    # Cleanup to prevent RAM crashes [cite: 440]
    del intersection, union, user_sums
    gc.collect()

    return user_jaccard_matrix



In [ ]:
# --- Execution ---
# Replace your cosine_similarity line with this:
user_similarity = compute_user_jaccard_similarity(train_data_matrix)

print(f"User-User Jaccard Matrix Shape: {user_similarity.shape}")

Calculating user intersection matrix...
Calculating user union matrix...
Computing final User Jaccard scores...
User-User Jaccard Matrix Shape: (7838, 7838)


In [ ]:
#Predict Positive Interactions Using User Similarity
# Define the function to predict interactions based on user similarity
def user_based_predict(interactions, similarity, epsilon=1e-9):
    """
    Predicts user-item interactions based on user-user similarity.
    Parameters:
        interactions (numpy array): The user-item interaction matrix.
        similarity (numpy array): The user-user similarity matrix.
        epsilon (float): Small constant added to the denominator to avoid division by zero.
    Returns:
        numpy array: The predicted interaction scores for each user-item pair.
    """
    # Calculate the weighted sum of interactions based on user similarity
    pred = similarity.dot(interactions) / (np.abs(similarity).sum(axis=1)[:, np.newaxis] + epsilon)
    return pred

user_prediction = user_based_predict(train_data_matrix, user_similarity)

In [ ]:
del user_similarity
gc.collect()

0

In [ ]:
# Precision
precision_user_k, recall_user_k = precision_recall_at_k(user_prediction, test_data_matrix, k=10)

# Get top 10 recommendations for user-based CF
user_based_recs_df = get_top_n_recommendations(user_prediction, n=10)
user_based_recs_df['Recommender_Type'] = 'User-Based'

#Submission Format
user_sub = create_submission_format(user_based_recs_df, 'user_cf_submission.csv')

precision_user_k

Successfully created: user_cf_submission.csv


np.float64(0.17270987496810017)

## History - already read

In [ ]:

def history_based_predict(train_matrix):
    """
    Assigns a fixed score to every book the user has already rented.
    This works because the test set is a subset of the training interactions.

    Parameters:
        train_matrix (numpy array): Binary user-item matrix (n_users x n_items).

    Returns:
        numpy array: Score matrix in float32.
    """
    # Simply convert the binary matrix to float32
    # Books read get a 1.0, books not read get a 0.0
    history_scores = train_matrix.astype(np.float32)

    return history_scores

# 1. Generate the history-based scores
history_prediction = history_based_predict(train_data_matrix)

# 2. Cleanup RAM
gc.collect()

print(f"History Prediction Matrix Created. Shape: {history_prediction.shape}")

History Prediction Matrix Created. Shape: (7838, 15291)


In [ ]:
precision_hist, recall_hist = precision_recall_at_k(history_prediction, test_data_matrix, k=10)


In [ ]:
precision_hist, recall_hist

(np.float64(0.051492727736669665), np.float64(0.24274826581627454))

## Subject with text analytics

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# 1. Prepare the text data
# Ensure the 'Subjects' column is treated as a string and handle missing values
items['Subjects'] = items['Subjects'].fillna('')

# 2. Text Representation using TF-IDF [cite: 1953, 1955]
# We use ngram_range=(1,2) to capture single words and 2-word phrases (e.g., "Social Science")
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
subject_vectors = tfidf.fit_transform(items['Subjects'])

# 3. Compute Item-to-Item Similarity [cite: 2008, 2010]
# This creates a matrix of how similar every book is to every other book based on subjects
subject_sim_matrix = cosine_similarity(subject_vectors)


In [ ]:
# 1. Use the item_based_predict logic from your lab to create a full matrix
# This calculates scores for every user based on their history and subject similarity
subject_prediction_matrix = item_based_predict(train_data_matrix, subject_sim_matrix)

del subject_sim_matrix
gc.collect()

# 2. Verify the shape is (n_users, n_items) [cite: 297, 885]
print(f"Prediction Matrix Shape: {subject_prediction_matrix.shape}")

# 3. Evaluate using your standard precision_recall_at_k function [cite: 198, 811]
precision_subject, recall_subject = precision_recall_at_k(subject_prediction_matrix, test_data_matrix, k=10)

# 4. Generate the Top-10 recommendations for your report [cite: 218, 1087]
subject_recs_df = get_top_n_recommendations(subject_prediction_matrix, n=10)
subject_recs_df['Recommender_Type'] = 'Subject-Based'

# 5. Create the submission file [cite: 232, 1136]
subject_sub = create_submission_format(subject_recs_df, 'subject_submission.csv')

print(f"Subject Recommender Precision@10: {precision_subject:.4f}")

Prediction Matrix Shape: (7838, 15291)
Successfully created: subject_submission.csv
Subject Recommender Precision@10: 0.0890


In [ ]:
print(f'Subject_recall{recall_subject}')

Subject_recall0.17230108730497234


## Titles with text analytics

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# 1. Prepare the text data
# Ensure the 'Title' column is treated as a string and handle missing values
items['Title'] = items['Title'].fillna('')

# 2. Text Representation using TF-IDF [cite: 1953, 1955]
# We use ngram_range=(1,2) to capture single words and 2-word phrases (e.g., "Social Science")
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
title_vectors = tfidf.fit_transform(items['Title'])

# 3. Compute Item-to-Item Similarity [cite: 2008, 2010]
# This creates a matrix of how similar every book is to every other book based on title
title_sim_matrix = cosine_similarity(title_vectors)


In [ ]:
# 1. Use the item_based_predict logic from your lab to create a full matrix
# This calculates scores for every user based on their title similarity
title_prediction_matrix = item_based_predict(train_data_matrix, title_sim_matrix)

del title_sim_matrix
gc.collect()

# 2. Verify the shape is (n_users, n_items) [cite: 297, 885]
print(f"Prediction Matrix Shape: {title_prediction_matrix.shape}")

# 3. Evaluate using your standard precision_recall_at_k function [cite: 198, 811]
precision_title, recall_title = precision_recall_at_k(title_prediction_matrix, test_data_matrix, k=10)

# 4. Generate the Top-10 recommendations for your report [cite: 218, 1087]
title_recs_df = get_top_n_recommendations(title_prediction_matrix, n=10)
title_recs_df['Recommender_Type'] = 'title-Based'

# 5. Create the submission file [cite: 232, 1136]
title_sub = create_submission_format(title_recs_df, 'title_submission.csv')

precision_title

Prediction Matrix Shape: (7838, 15291)
Successfully created: title_submission.csv


np.float64(0.10441439142639494)

In [ ]:
recall_title

np.float64(0.21044628972651533)

# Sequential pattern in the data

In [ ]:
import numpy as np
import gc

def sequential_pattern_predict(train_matrix):
    """
    Exploits a forward sequential bias for the first 3000 users.
    Identifies the highest ItemID in the user's history and recommends the next two IDs.

    Parameters:
        train_matrix (numpy array): Binary user-item matrix (n_users x n_items).

    Returns:
        numpy array: Score matrix in float32 for hybrid compatibility.
    """
    n_users, n_items = train_matrix.shape
    # Initialize with zeros in float32 to save RAM
    pred_matrix = np.zeros((n_users, n_items), dtype=np.float32)

    # Limit logic specifically to the first 3000 users as requested
    for u in range(min(3000, n_users)):
        # Find all ItemIDs the user has already rented
        interacted_ids = np.where(train_matrix[u] > 0)[0]

        if len(interacted_ids) > 0:
            # Anchor at the highest ID found in the training set
            max_id = interacted_ids.max()

            # Recommendation Rule: Recommend the two next books (i+1, i+2)
            # Example: [100, 232, 234, 235] -> recommend 236 and 237
            for step in [1, 2]:
                target_item = max_id + step

                # Bounds check to stay within the library size (15291 items)
                if target_item < n_items:
                    # Assign a high score to these specific two items
                    pred_matrix[u, target_item] = 1.0

    return pred_matrix

# 1. Generate the Sequential Bias scores with the new 3000-user limit
sequential_prediction = sequential_pattern_predict(train_data_matrix)

# 2. Memory Management: Clean up
gc.collect()

print(f"Sequential Bias Matrix (Forward, First 3000 Users) Created.")
print(f"Matrix Shape: {sequential_prediction.shape}")

Sequential Bias Matrix (Forward, First 3000 Users) Created.
Matrix Shape: (7838, 15291)


In [ ]:
# 3. Evaluate using your standard precision_recall_at_k function [cite: 198, 811]
precision_sequential, recall_sequential = precision_recall_at_k(sequential_prediction, test_data_matrix, k=10)

# 4. Generate the Top-10 recommendations for your report [cite: 218, 1087]
sequential_recs_df = get_top_n_recommendations(sequential_prediction, n=10)
sequential_recs_df['Recommender_Type'] = 'sequential'

# 5. Create the submission file [cite: 232, 1136]
sequential_sub = create_submission_format(sequential_recs_df, 'sequential_submission.csv')

precision_sequential, recall_sequential

Successfully created: sequential_submission.csv


(np.float64(0.00019137535085480992), np.float64(0.0010889275332485635))

# Hybrid model 2

In [ ]:
# --- Hybrid Recommender System --- user/item

# 1. Define the weights for each model (should sum to 1.0)
# You can adjust these based on which model performed better

total_before = 1-0.22

history_w = 0.25
user_w = 0.3
item_w = 0.22
subject_w = 0.02
title_w = 0.076
sequential_w = 0.134



# 2. Combine the predictions
# Both matrices have the same shape (n_users, n_items)
best_hybrid_prediction = \
 (user_w * user_prediction) + \
  (item_w * item_prediction) + \
   (subject_w * subject_prediction_matrix) + \
    (history_w * history_prediction) + \
     (title_w * title_prediction_matrix) + \
      (sequential_w * sequential_prediction)


# 3. Get top 10 recommendations for the hybrid model
best_hybrid_recs_df = get_top_n_recommendations(best_hybrid_prediction, n=10)
best_hybrid_recs_df['Recommender_Type'] = 'Best Hybrid'

best_Hybrid_sub = create_submission_format(best_hybrid_recs_df, 'best_hybrid_submission.csv')

precision_best_hybrid, recall_best_hybrid = precision_recall_at_k(best_hybrid_prediction, test_data_matrix, k=10)



Successfully created: best_hybrid_submission.csv


In [ ]:
print(f'history_w:{history_w}, publisher_w:{publisher_w}, combined_text_w:{combined_text_w}, user_w:{user_w}, item_w:{item_w}, subject_w:{subject_w}, title_w:{title_w}, sequential_w:{sequential_w}')

history_w:0.25, publisher_w:0, combined_text_w:0, user_w:0.2980769230769231, item_w:0.22115384615384615, subject_w:0.019230769230769232, title_w:0.07692307692307693, sequential_w:0.1346153846153846


In [ ]:
del item_prediction
del user_prediction
del history_prediction
del combined_text_prediction
del subject_prediction_matrix
del title_prediction_matrix
del sequential_prediction
gc.collect()

30

# Cross Validation

## Item-Item

In [ ]:
# item-item Cross Validation loop with Jaccard Similarity

from sklearn.model_selection import KFold
import numpy as np
import gc

# 1. Setup the K-Fold parameters
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Lists to store the results of each fold
cv_precision = []
cv_recall = []

print(f"Starting {n_splits}-fold Cross-Validation for Jaccard Item-Based CF...")

# 2. The Cross-Validation Loop
for fold, (train_idx, test_idx) in enumerate(kf.split(interactions)):
    print(f"\n--- Processing Fold {fold+1}/{n_splits} ---")

    # Create fold-specific data frames
    train_df = interactions.iloc[train_idx]
    test_df = interactions.iloc[test_idx]

    # Generate user-item matrices for this specific fold
    train_matrix = create_data_matrix(train_df, n_users, n_items)
    test_matrix = create_data_matrix(test_df, n_users, n_items)

    # Calculate Item-Item Similarity using your custom Jaccard function
    print("Computing Jaccard item similarity...")
    item_sim = compute_jaccard_similarity(train_matrix)

    # 3. Predict using your item-based prediction function
    print("Generating predictions...")
    item_pred = item_based_predict(train_matrix, item_sim)

    # 4. Evaluate this fold
    print("Evaluating metrics...")
    prec, rec = precision_recall_at_k(item_pred, test_matrix, k=10)
    cv_precision.append(prec)
    cv_recall.append(rec)

    print(f"Fold {fold+1} complete: Precision@10: {prec:.4f}, Recall@10: {rec:.4f}")

    # 5. RAM Management: Clean up large matrices immediately to prevent crashes
    del train_matrix, test_matrix, item_sim, item_pred
    gc.collect()

# 6. Output Final Results for your README Table
print("\n=======================================")
print("--- Final Jaccard CV Results ---")
print(f"Average Precision@10: {np.mean(cv_precision):.4f}")
print(f"Average Recall@10:    {np.mean(cv_recall):.4f}")
print("=======================================")

Starting 5-fold Cross-Validation for Jaccard Item-Based CF...

--- Processing Fold 1/5 ---
Computing Jaccard item similarity...
Calculating intersection matrix...
Calculating union matrix...
Computing final Jaccard scores...
Generating predictions...
Evaluating metrics...
Fold 1 complete: Precision@10: 0.0496, Recall@10: 0.2149

--- Processing Fold 2/5 ---
Computing Jaccard item similarity...
Calculating intersection matrix...
Calculating union matrix...
Computing final Jaccard scores...
Generating predictions...
Evaluating metrics...
Fold 2 complete: Precision@10: 0.0497, Recall@10: 0.2161

--- Processing Fold 3/5 ---
Computing Jaccard item similarity...
Calculating intersection matrix...
Calculating union matrix...
Computing final Jaccard scores...
Generating predictions...
Evaluating metrics...
Fold 3 complete: Precision@10: 0.0500, Recall@10: 0.2205

--- Processing Fold 4/5 ---
Computing Jaccard item similarity...
Calculating intersection matrix...
Calculating union matrix...
Compu

## User-User

In [ ]:
# user-user Cross Validation loop with Jaccard Similarity

from sklearn.model_selection import KFold
import numpy as np
import gc

# 1. Setup the K-Fold parameters
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Lists to store the results of each fold
cv_precision = []
cv_recall = []

print(f"Starting {n_splits}-fold Cross-Validation for Jaccard User-Based CF...")

# 2. The Cross-Validation Loop
for fold, (train_idx, test_idx) in enumerate(kf.split(interactions)):
    print(f"\n--- Processing Fold {fold+1}/{n_splits} ---")

    # Create fold-specific data frames
    train_df = interactions.iloc[train_idx]
    test_df = interactions.iloc[test_idx]

    # Generate user-item matrices for this specific fold
    train_matrix = create_data_matrix(train_df, n_users, n_items)
    test_matrix = create_data_matrix(test_df, n_users, n_items)

    # Calculate User-User Similarity using your custom Jaccard function
    print("Computing Jaccard user similarity...")
    user_sim = compute_user_jaccard_similarity(train_matrix)

    # 3. Predict using your user-based prediction function
    print("Generating predictions...")
    user_pred = user_based_predict(train_matrix, user_sim)

    # 4. Evaluate this fold
    print("Evaluating metrics...")
    prec, rec = precision_recall_at_k(user_pred, test_matrix, k=10)
    cv_precision.append(prec)
    cv_recall.append(rec)

    print(f"Fold {fold+1} complete: Precision@10: {prec:.4f}, Recall@10: {rec:.4f}")

    # 5. RAM Management: Clean up large matrices immediately to prevent crashes
    del train_matrix, test_matrix, user_sim, user_pred
    gc.collect()

# 6. Output Final Results for your README Table
print("\n=======================================")
print("--- Final Jaccard User-User CV Results ---")
print(f"Average Precision@10: {np.mean(cv_precision):.4f}")
print(f"Average Recall@10:    {np.mean(cv_recall):.4f}")
print("=======================================")

## History

In [ ]:
# History Recommender Cross Validation loop

from sklearn.model_selection import KFold
import numpy as np
import gc

# 1. Setup the K-Fold parameters
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Lists to store the results of each fold
cv_precision = []
cv_recall = []

print(f"Starting {n_splits}-fold Cross-Validation for History-Based Recommender...")

# 2. The Cross-Validation Loop
for fold, (train_idx, test_idx) in enumerate(kf.split(interactions)):
    print(f"\n--- Processing Fold {fold+1}/{n_splits} ---")

    # Create fold-specific data frames
    train_df = interactions.iloc[train_idx]
    test_df = interactions.iloc[test_idx]

    # Generate user-item matrices for this specific fold
    train_matrix = create_data_matrix(train_df, n_users, n_items)
    test_matrix = create_data_matrix(test_df, n_users, n_items)

    # 3. Predict using your history_based_predict function
    print("Generating history-based scores...")
    history_pred = history_based_predict(train_matrix)

    # 4. Evaluate this fold
    print("Evaluating metrics...")
    prec, rec = precision_recall_at_k(history_pred, test_matrix, k=10)
    cv_precision.append(prec)
    cv_recall.append(rec)

    print(f"Fold {fold+1} complete: Precision@10: {prec:.4f}, Recall@10: {rec:.4f}")

    # 5. RAM Management: Clean up fold matrices immediately
    del train_matrix, test_matrix, history_pred
    gc.collect()

# 6. Output Final Results for your README Table
print("\n=======================================")
print("--- Final History Recommender CV Results ---")
print(f"Average Precision@10: {np.mean(cv_precision):.4f}")
print(f"Average Recall@10:    {np.mean(cv_recall):.4f}")
print("=======================================")

## Subject

In [ ]:
# Subject Text Analytics Cross Validation loop

from sklearn.model_selection import KFold
import numpy as np
import gc

# 1. Setup the K-Fold parameters
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Lists to store the results of each fold
cv_precision = []
cv_recall = []

# 2. Pre-compute the Item-Item Subject Similarity Matrix ONCE to save RAM and time
print("Pre-computing global subject similarity matrix...")
items['Subjects'] = items['Subjects'].fillna('')
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
subject_vectors = tfidf.fit_transform(items['Subjects'])
global_subject_sim = cosine_similarity(subject_vectors).astype(np.float32) # float32 saves 50% RAM

# Clear intermediate matrix vector representations to free memory
del subject_vectors
gc.collect()

print(f"Starting {n_splits}-fold Cross-Validation for Subject Text Analytics...")

# 3. The Cross-Validation Loop
for fold, (train_idx, test_idx) in enumerate(kf.split(interactions)):
    print(f"\n--- Processing Fold {fold+1}/{n_splits} ---")

    # Create fold-specific data frames
    train_df = interactions.iloc[train_idx]
    test_df = interactions.iloc[test_idx]

    # Generate user-item matrices for this specific fold
    train_matrix = create_data_matrix(train_df, n_users, n_items)
    test_matrix = create_data_matrix(test_df, n_users, n_items)

    # 4. Predict using the fold's train history and the pre-computed text similarities
    print("Generating subject-based predictions for this fold...")
    subject_pred = item_based_predict(train_matrix, global_subject_sim)

    # 5. Evaluate this fold
    print("Evaluating metrics...")
    prec, rec = precision_recall_at_k(subject_pred, test_matrix, k=10)
    cv_precision.append(prec)
    cv_recall.append(rec)

    print(f"Fold {fold+1} complete: Precision@10: {prec:.4f}, Recall@10: {rec:.4f}")

    # 6. RAM Management: Clean up fold-specific arrays
    del train_matrix, test_matrix, subject_pred
    gc.collect()

# Final cleanup of the big similarity matrix
del global_subject_sim
gc.collect()

# 7. Output Final Results for your README Table 2
print("\n=======================================")
print("--- Final Subject Text Analytics CV Results ---")
print(f"Average Precision@10: {np.mean(cv_precision):.4f}")
print(f"Average Recall@10:    {np.mean(cv_recall):.4f}")
print("=======================================")

## Titles

In [ ]:
# Title Text Analytics Cross Validation loop

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold
import numpy as np
import gc

# 1. Setup the K-Fold parameters
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Lists to store the results of each fold
cv_precision = []
cv_recall = []

# 2. Pre-compute the Item-Item Title Similarity Matrix ONCE to save RAM and time
print("Pre-computing global title similarity matrix...")
items['Title'] = items['Title'].fillna('')
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
title_vectors = tfidf.fit_transform(items['Title'])
global_title_sim = cosine_similarity(title_vectors).astype(np.float32) # float32 halves RAM usage

# Clear intermediate vector representations to save memory
del title_vectors
gc.collect()

print(f"Starting {n_splits}-fold Cross-Validation for Title Text Analytics...")

# 3. The Cross-Validation Loop
for fold, (train_idx, test_idx) in enumerate(kf.split(interactions)):
    print(f"\n--- Processing Fold {fold+1}/{n_splits} ---")

    # Create fold-specific data frames
    train_df = interactions.iloc[train_idx]
    test_df = interactions.iloc[test_idx]

    # Generate user-item matrices for this specific fold
    train_matrix = create_data_matrix(train_df, n_users, n_items)
    test_matrix = create_data_matrix(test_df, n_users, n_items)

    # 4. Predict using the fold's train history and the pre-computed title similarities
    print("Generating title-based predictions for this fold...")
    title_pred = item_based_predict(train_matrix, global_title_sim)

    # 5. Evaluate this fold
    print("Evaluating metrics...")
    prec, rec = precision_recall_at_k(title_pred, test_matrix, k=10)
    cv_precision.append(prec)
    cv_recall.append(rec)

    print(f"Fold {fold+1} complete: Precision@10: {prec:.4f}, Recall@10: {rec:.4f}")

    # 6. RAM Management: Clean up fold-specific arrays
    del train_matrix, test_matrix, title_pred
    gc.collect()

# Final cleanup of the big similarity matrix
del global_title_sim
gc.collect()

# 7. Output Final Results for your README Table 2
print("\n=======================================")
print("--- Final Title Text Analytics CV Results ---")
print(f"Average Precision@10: {np.mean(cv_precision):.4f}")
print(f"Average Recall@10:    {np.mean(cv_recall):.4f}")
print("=======================================")

## Data Bias

In [ ]:
# Sequential Bias Recommender Cross Validation loop

from sklearn.model_selection import KFold
import numpy as np
import gc

# 1. Setup the K-Fold parameters
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Lists to store the results of each fold
cv_precision = []
cv_recall = []

print(f"Starting {n_splits}-fold Cross-Validation for Sequential Pattern Recommender...")

# 2. The Cross-Validation Loop
for fold, (train_idx, test_idx) in enumerate(kf.split(interactions)):
    print(f"\n--- Processing Fold {fold+1}/{n_splits} ---")

    # Create fold-specific data frames
    train_df = interactions.iloc[train_idx]
    test_df = interactions.iloc[test_idx]

    # Generate user-item matrices for this specific fold
    train_matrix = create_data_matrix(train_df, n_users, n_items)
    test_matrix = create_data_matrix(test_df, n_users, n_items)

    # 3. Predict using your sequential_pattern_predict function
    print("Generating forward-sequential bias scores...")
    sequential_pred = sequential_pattern_predict(train_matrix)

    # 4. Evaluate this fold
    print("Evaluating metrics...")
    prec, rec = precision_recall_at_k(sequential_pred, test_matrix, k=10)
    cv_precision.append(prec)
    cv_recall.append(rec)

    print(f"Fold {fold+1} complete: Precision@10: {prec:.4f}, Recall@10: {rec:.4f}")

    # 5. RAM Management: Clean up fold matrices immediately
    del train_matrix, test_matrix, sequential_pred
    gc.collect()

# 6. Output Final Results for your README Table 2
print("\n=======================================")
print("--- Final Sequential Bias CV Results ---")
print(f"Average Precision@10: {np.mean(cv_precision):.4f}")
print(f"Average Recall@10:    {np.mean(cv_recall):.4f}")
print("=======================================")